# 🧱 Part 2 심화: ResNet Block과 ONNX 변환

---

## 🎯 학습 목표
| # | 목표 | 핵심 개념 |
|---|------|----------|
| 1 | Skip Connection 이해 | Residual Learning |
| 2 | Projection Shortcut | 차원 맞추기 (1x1 Conv) |
| 3 | 복잡한 모델 ONNX 변환 | BatchNorm, Add 연산 |
| 4 | ResNet 교차 검증 | 동일 결과 확인 |


---

# ❓ Skip Connection 특화 Q&A (상세 해설)

---

## Q1: Skip Connection(잔차 연결)이 뭔가요?

Skip Connection은 **입력을 출력에 직접 더하는 연결 방식**입니다.  
즉, 네트워크가 단순히 입력을 변환한 결과만 내놓는 게 아니라, **원래 입력을 그대로 더해주는 구조**예요.

### 직관적 이해
- 일반적인 신경망: 입력 \(x\) → 여러 층을 거쳐 변환 \(F(x)\) → 출력  
- Skip Connection: 출력 = \(F(x) + x\)  
  → 변환된 값과 원래 입력을 합쳐서 최종 결과를 만듭니다.

```text
x → [Conv → BN → ReLU → Conv → BN] → (+) → ReLU → out
|                                     ↑
+───────── (identity) ────────────────+
```

### 왜 좋은가?
- **기울기 소실 완화**: 깊은 네트워크에서는 역전파 시 기울기가 점점 작아져서 학습이 멈추는 문제가 생깁니다. Skip Connection은 입력을 직접 더해주므로 기울기가 "지름길"을 통해 원래 입력까지 잘 전달됩니다.
- **더 깊은 네트워크 학습 가능**: ResNet이 수백 층까지도 학습 가능한 이유가 바로 이 구조 덕분입니다.
- **변화량만 학습**: 네트워크는 "입력 전체"를 새로 학습할 필요 없이, 입력 대비 "얼마나 바꿀지(Residual)"만 학습하면 됩니다. 즉, 학습 난이도가 크게 줄어듭니다.

👉 비유: 글을 고칠 때 원문을 다 지우고 새로 쓰는 대신, 원문을 그대로 두고 필요한 부분만 수정하는 방식과 같습니다.

---

## Q2: Skip Connection이 ONNX에서 왜 문제가 되나요?

ONNX 변환 시 Skip Connection이 문제가 되는 가장 큰 이유는 **텐서 형상(Shape) 불일치**입니다.

### 왜 불일치가 생기나?
- Skip Connection은 입력 \(x\)와 변환된 출력 \(F(x)\)를 더해야 합니다.  
- 하지만 두 텐서의 **채널 수**나 **공간 크기(Height, Width)**가 다르면 덧셈이 불가능합니다.  
- 예: 입력은 (Batch, 64, 28, 28), 출력은 (Batch, 128, 14, 14) → 그냥 더하면 에러!

### ONNX에서 특히 까다로운 이유
- PyTorch에서는 내부적으로 브로드캐스팅이나 자동 변환이 어느 정도 허용되지만,  
- ONNX는 **고정된 연산 그래프**를 내보내야 하므로, Shape이 조금이라도 다르면 변환 자체가 실패하거나 잘못된 그래프가 만들어집니다.

👉 따라서 Skip Connection을 쓰려면 반드시 **입력과 출력의 Shape을 맞추는 장치**가 필요합니다.

---

## Q3: Projection Shortcut은 언제 필요한가요?

Projection Shortcut은 Skip Connection에서 입력과 출력의 Shape이 다를 때, **1x1 Convolution**을 사용해 차원을 맞추는 방법입니다.

### 필요한 경우
1. **채널 수 변경**  
   - 예: 입력 채널 64 → 출력 채널 128  
   - 그냥 더하면 불가능 → 1x1 Conv로 입력 채널을 128로 변환 후 더함.

2. **크기 변경 (stride ≠ 1)**  
   - 예: stride=2로 다운샘플링 → 입력 크기 28x28 → 출력 크기 14x14  
   - 크기가 다르면 덧셈 불가 → 1x1 Conv + stride=2로 입력 크기를 줄여서 맞춤.

### 코드 예시
```python
if stride != 1 or in_channels != out_channels:
    # 1x1 Conv로 차원 맞추기
    self.shortcut = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride)
else:
    # 그냥 identity 연결
    self.shortcut = nn.Identity()
```

👉 Projection Shortcut은 Skip Connection의 "어댑터" 역할을 합니다.  
비유하자면, 서로 다른 규격의 전기 플러그를 연결할 때 변환 어댑터를 쓰는 것과 같습니다.

---

## Q4: BatchNorm은 ONNX에서 어떻게 처리되나요?

BatchNorm은 학습 모드와 추론 모드에서 동작 방식이 다릅니다.  
ONNX 변환 시 반드시 **`eval()` 모드**로 설정해야 하는 이유가 여기 있습니다.

### 학습 모드 (`train()`)
- 각 배치마다 평균과 분산을 새로 계산합니다.  
- 즉, 입력 데이터에 따라 BatchNorm 결과가 매번 달라집니다.

### 추론 모드 (`eval()`)
- 학습 과정에서 누적된 **이동평균(mean, variance)**을 사용합니다.  
- 따라서 입력 데이터가 달라도 BatchNorm 결과가 안정적으로 동일하게 나옵니다.

### ONNX 변환 시 문제
- 만약 `model.train()` 상태에서 변환하면, BatchNorm이 "배치별 계산"으로 기록됩니다.  
- 이는 ONNX 그래프에서 재현 불가능하거나, 추론 시 결과가 달라지는 문제를 일으킵니다.  
- 따라서 반드시 `model.eval()`을 호출해 BatchNorm을 **고정된 추론 모드**로 바꾼 뒤 변환해야 합니다.

👉 비유: 식당에서 요리 레시피를 기록할 때, "손님마다 간을 다르게 한다"는 방식으로 적어두면 재현 불가능합니다. 대신 "간은 평균값으로 고정한다"라고 적어야 어디서든 같은 맛을 낼 수 있습니다.

---

# 🎯 최종 요약

- **Skip Connection**: 입력을 출력에 직접 더하는 구조 → 기울기 소실 완화, 깊은 네트워크 학습 가능.  
- **문제점**: ONNX 변환 시 Shape 불일치가 발생하면 덧셈 불가.  
- **해결책**: Projection Shortcut(1x1 Conv)으로 채널/크기를 맞춰줌.  
- **BatchNorm**: 반드시 `eval()` 모드에서 변환해야 안정적인 추론 그래프가 생성됨.  

👉 결국 Skip Connection을 ONNX로 안전하게 변환하려면 **Shape 맞추기 + BatchNorm 고정**이라는 두 가지 핵심 규칙을 지켜야 합니다.

---


---
## 🔧 ZONE 1: 라이브러리 임포트

In [7]:
# ═══════════════════════════════════════════════════════════════════
# 📦 [라이브러리 임포트]
# ═══════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.onnx
import onnx
import onnxruntime as ort
import numpy as np
import os

# ═══════════════════════════════════════════════════════════════════
# 🎨 [유틸리티 함수]
# ═══════════════════════════════════════════════════════════════════
def print_header(title):
    print(f"\n{'='*70}")
    print(f"{title}")
    print(f"{'='*70}")

# 📂 폴더 생성
os.makedirs('saved_models', exist_ok=True)

print("✅ 라이브러리 임포트 완료!")
print(f"   PyTorch 버전: {torch.__version__}")

✅ 라이브러리 임포트 완료!
   PyTorch 버전: 2.10.0.dev20251029+cu130


---
## 🧱 ZONE 2: Residual Block 정의

### 💡 ResNet Basic Block 구조
```
x ──────────────────────────────────┐
│                                   │
├─→ Conv3x3 → BN → ReLU             │
│            ↓                      │
├─→ Conv3x3 → BN                    │
│            ↓                      │
│         ↓                         │ (identity or projection)
│        (+) ←──────────────────────┘
│         ↓
└─→     ReLU
          ↓
        output
```

In [ ]:
print_header("🧱 [Step 1] Residual Block 정의")

# ═══════════════════════════════════════════════════════════════════
# 📐 [ResidualBlock 클래스 정의]
# ═══════════════════════════════════════════════════════════════════
#
# [역할] ResNet의 'Basic Block' 구현
#   - 3x3 Conv 2개 + Skip Connection
#   - 필요시 Projection Shortcut 적용
#
class ResidualBlock(nn.Module):
    """
    ResNet Basic Block
    
    [구조]
    x → [conv1 → bn1 → relu → conv2 → bn2] → (+) → relu → out
    |                                         ↑
    +------ (identity or projection) ---------+
    """
    
    def __init__(self, in_channels, out_channels, stride=1):
        """
        [인자 설명]
          - in_channels: 입력 채널 수
          - out_channels: 출력 채널 수
          - stride: 다운샘플링 비율 (1=유지, 2=절반)
        """
        super(ResidualBlock, self).__init__()
        
        # ┌─────────────────────────────────────────────────────────────
        # │ 🔵 Main Path (메인 경로: F(x))
        # └─────────────────────────────────────────────────────────────
        
        # Conv1: 3x3, stride 적용
        # [기계적 구동] bias=False를 쓰는 이유: 
        # 바로 뒤에 오는 BatchNorm이 'beta(편향)' 파라미터를 가지고 있어서 Conv의 bias는 중복 연산이 됩니다.
        self.conv1 = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=3, stride=stride, padding=1, bias=False
        )
        
        # BatchNorm1: 채널별 정규화 (입력 분포를 평균 0, 분산 1로 맞춤)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # ReLU: 비선형 활성화
        # [기계적 구동] inplace=True: 
        # 메모리를 아끼기 위해 새로운 텐서를 만들지 않고 입력 텐서 값을 직접 수정합니다.
        self.relu = nn.ReLU(inplace=True)
        
        # Conv2: 3x3, stride=1 고정 (이미 Conv1에서 크기를 줄였으므로 여기선 유지)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels,
            kernel_size=3, stride=1, padding=1, bias=False
        )
        
        # BatchNorm2
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # ┌─────────────────────────────────────────────────────────────
        # │ 🟡 Shortcut Path (지름길 경로: x)
        # └─────────────────────────────────────────────────────────────
        #
        # [기본] Identity: 아무것도 안 하고 입력 x를 그대로 통과시킴
        # nn.Sequential() 빈 껍데기는 입력을 그대로 반환합니다.
        self.shortcut = nn.Sequential()
        
        # ┌─────────────────────────────────────────────────────────────
        # │ 🔧 Projection Shortcut 조건
        # │
        # │ [언제?] 다음 중 하나라도 해당되면 단순 덧셈(x + F(x))이 불가능함:
        # │   1. stride != 1: 이미지 크기가 줄어듬 (예: 28x28 + 14x14 -> 불가 ❌)
        # │   2. in_ch != out_ch: 채널 수가 다름 (예: 16장 + 32장 -> 불가 ❌)
        # │
        # │ [해결] 1x1 Conv로 x의 모양(Shape)을 F(x)와 똑같이 강제로 맞춤
        # │   - kernel_size=1: 픽셀 하나하나의 값만 채널 축으로 섞어줌 (공간 정보 유지)
        # └─────────────────────────────────────────────────────────────
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels,
                    kernel_size=1, stride=stride, bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        # 🔒 원본 입력 저장 (나중에 더하기 위해 Backup)
        identity = x
        
        # ----- Main Path (학습 대상) -----
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # ----- Skip Connection (핵심) -----
        # ➕ 원본(또는 Projection된 원본) + 변형된 값
        # [기계적 구동] 파이토치의 `+` 연산자는 ONNX 변환 시 `Add` 노드로 번역됩니다.
        # 이때 두 텐서의 Shape이 정확히 일치하지 않으면 ONNX 변환 에러가 나거나 실행 시 죽습니다.
        out += self.shortcut(identity)
        
        # 마지막 ReLU: 덧셈 후에 비선형성을 가함
        out = self.relu(out)
        
        return out


# ═══════════════════════════════════════════════════════════════════
# 📊 [블록 테스트]
# ═══════════════════════════════════════════════════════════════════
print("[Identity Shortcut 테스트]")
print("   - 입력: 16채널, 출력: 16채널, stride=1")
block_identity = ResidualBlock(16, 16, stride=1)
x = torch.randn(1, 16, 28, 28)
# shape 검증: 입력과 출력이 같아야 함
print(f"   - 입력 Shape: {x.shape}")
print(f"   - 출력 Shape: {block_identity(x).shape}")
print(f"   - Shortcut 타입: Identity (빈 Sequential)")

print("\n[Projection Shortcut 테스트]")
print("   - 입력: 16채널, 출력: 32채널, stride=2 (다운샘플링)")
block_projection = ResidualBlock(16, 32, stride=2)
# shape 검증: 크기는 반토막(14), 채널은 2배(32)가 되어야 함
print(f"   - 입력 Shape: {x.shape}")
print(f"   - 출력 Shape: {block_projection(x).shape}")
print(f"   - Shortcut 타입: 1x1 Conv + BN")


🧱 [Step 1] Residual Block 정의
[Identity Shortcut 테스트]
   - 입력: 16채널, 출력: 16채널, stride=1
   - 입력 Shape: torch.Size([1, 16, 28, 28])
   - 출력 Shape: torch.Size([1, 16, 28, 28])
   - Shortcut 타입: Identity (빈 Sequential)

[Projection Shortcut 테스트]
   - 입력: 16채널, 출력: 32채널, stride=2 (다운샘플링)
   - 입력 Shape: torch.Size([1, 16, 28, 28])
   - 출력 Shape: torch.Size([1, 32, 14, 14])
   - Shortcut 타입: 1x1 Conv + BN


### 📊 결과 해석
- **Identity**: 채널/크기 동일 → 입력을 그대로 더함
- **Projection**: 채널 16→32, 크기 28→14 → 1x1 Conv로 맞춤

**핵심**: Skip Connection이 가능하려면 **Shape이 일치**해야 함!

---
## 🏗️ ZONE 3: MiniResNet 모델 정의

In [9]:
print_header("🏗️ [Step 2] MiniResNet 모델 정의")

# ═══════════════════════════════════════════════════════════════════
# 📐 [MiniResNet 클래스 정의]
# ═══════════════════════════════════════════════════════════════════
#
# [역할] ResNet-18을 MNIST(28x28)에 맞게 축소한 모델
#   - 3개의 Residual Block을 직렬로 연결
#   - Global Average Pooling
#   - 최종 분류기
#
class MiniResNet(nn.Module):
    """
    MNIST용 축소 ResNet
    
    [구조]
    (1, 28, 28) → Conv → BN → ReLU
         ↓
    Layer1 (16, 28, 28) - Identity
         ↓
    Layer2 (32, 14, 14) - Projection + Downsample
         ↓
    Layer3 (64, 7, 7)   - Projection + Downsample
         ↓
    Global AvgPool (64, 1, 1)
         ↓
    FC (10)
    """
    
    def __init__(self, num_classes=10):
        super(MiniResNet, self).__init__()
        
        # ┌─────────────────────────────────────────────────────────────
        # │ 🔵 초기 레이어 (Stem)
        # │
        # │ [역할] 입력 채널(1) → 기본 채널(16) 확장
        # └─────────────────────────────────────────────────────────────
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)
        
        # ┌─────────────────────────────────────────────────────────────
        # │ 🧱 Residual Blocks
        # └─────────────────────────────────────────────────────────────
        
        # Layer1: 16 → 16, stride=1 크기와 채널 유지 (특징 추출 강화) 
        # [결과] (N, 16, 28, 28) → (N, 16, 28, 28)
        self.layer1 = ResidualBlock(16, 16, stride=1)
        
        # Layer2: 16 → 32, stride=2 (다운샘플링) 크기 절반(14x14), 채널 2배(32)
        # [결과] (N, 16, 28, 28) → (N, 32, 14, 14)
        self.layer2 = ResidualBlock(16, 32, stride=2)
        
        # Layer3: 32 → 64, stride=2 (다운샘플링) 크기 절반(7x7), 채널 2배(64)
        # [결과] (N, 32, 14, 14) → (N, 64, 7, 7)
        self.layer3 = ResidualBlock(32, 64, stride=2)
        
        # ┌─────────────────────────────────────────────────────────────
        # │ 🎯 Global Average Pooling
        # │
        # │ [문법] nn.AdaptiveAvgPool2d((H, W))
        # │   - 입력 크기에 상관없이 (H, W) 크기로 출력
        # │   - (1, 1)로 설정하면 채널별 평균값 1개씩
        # │
        # │ [장점] 입력 이미지 크기가 달라도 동작!
        # └─────────────────────────────────────────────────────────────
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # 📊 최종 분류기: 64개 특징을 보고 0~9 숫자 맞추기
        # [결과] (N, 64) → (N, 10)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        # 1️⃣ Stem
        # (N, 1, 28, 28) → (N, 16, 28, 28)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        
        # 2️⃣ Residual Blocks (여기서 Add 연산이 수두룩하게 일어남)
        x = self.layer1(x)  # (N, 16, 28, 28)
        x = self.layer2(x)  # (N, 32, 14, 14)
        x = self.layer3(x)  # (N, 64, 7, 7)
        
        # 3️⃣ Global Average Pooling
        # [기계적 구동] 7x7 픽셀 49개의 평균을 구함 -> (N, 64, 1, 1)
        # (N, 64, 7, 7) → (N, 64, 1, 1)
        x = self.avgpool(x)
        
        # 4️⃣ Flatten
        # [기계적 구동] (N, 64, 1, 1) 4차원 텐서를 (N, 64) 2차원 행렬로 폄
        # (N, 64, 1, 1) → (N, 64)
        x = torch.flatten(x, 1)
        
        # 5️⃣ 최종 분류
        # (N, 64) → (N, 10)
        x = self.fc(x)
        
        return x


# ═══════════════════════════════════════════════════════════════════
# 🏭 [모델 생성]
# ═══════════════════════════════════════════════════════════════════
resnet_model = MiniResNet(num_classes=10)

# [중요] 평가 모드 전환 (BatchNorm 고정)
# 이것을 안 하면 ONNX Export 과정에서 "학습용 그래프(통계 계산 포함)"가 그려져서
# 모델이 엄청 복잡해지거나 특정 런타임에서 에러가 납니다.
resnet_model.eval()

# 파라미터 수 계산
total_params = sum(p.numel() for p in resnet_model.parameters())

print(f"✅ MiniResNet 생성 완료!")
print(f"   총 파라미터: {total_params:,} 개")
print(f"\n[모델 구조]")
print(resnet_model)


🏗️ [Step 2] MiniResNet 모델 정의
✅ MiniResNet 생성 완료!
   총 파라미터: 77,754 개

[모델 구조]
MiniResNet(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): ResidualBlock(
    (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (shortcut): Sequential()
  )
  (layer2): ResidualBlock(
    (conv1): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (con

### 📊 결과 해석
- **Layer1**: Identity Shortcut (채널/크기 동일)
- **Layer2**: Projection Shortcut (16→32, 28→14)
- **Layer3**: Projection Shortcut (32→64, 14→7)

**SimpleCNN 대비**: Skip Connection으로 더 깊은 구조 가능!

---
## 🔄 ZONE 4: ResNet ONNX 변환

In [10]:
print_header("🔄 [Step 3] ResNet ONNX 변환")

# ═══════════════════════════════════════════════════════════════════
# 📦 [ONNX Export 설정]
# ═══════════════════════════════════════════════════════════════════

RESNET_ONNX_PATH = 'saved_models/mini_resnet.onnx'

# 더미 입력 생성 (경로 추적용)
dummy_input = torch.randn(1, 1, 28, 28)

print("ONNX Export 시도 중...")
print("   - Skip Connection (덧셈 연산) 포함")
print("   - BatchNorm (eval 모드) 포함")
print("   - Projection Shortcut (1x1 Conv) 포함\n")

try:
    # ═══════════════════════════════════════════════════════════════
    # 📤 [ONNX Export 수행]
    # ═══════════════════════════════════════════════════════════════
    #
    # [기계적 구동]
    # 1. Tracing: dummy_input을 모델에 넣고 흘려보냅니다.
    # 2. Graph Construction: 연산(Node)과 데이터(Tensor)를 기록합니다.
    # 3. Skip Connection 처리: `out += shortcut` 코드를 만나는 순간
    #    ONNX 그래프에 `Add` 노드를 생성하고, 두 입력 엣지를 연결합니다.
    #
    torch.onnx.export(
        resnet_model,
        dummy_input,
        RESNET_ONNX_PATH,
        export_params=True,
        opset_version=13, # ResNet 변환에는 11 이상 권장 (Add, BN 최적화)
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        },
        # ⚠️ Windows/PyTorch 2.x 호환성: Dynamo 컴파일러 비활성화 (안전 모드)
        dynamo=False,
    )
    
    print(f"✅ ResNet ONNX Export 성공: {RESNET_ONNX_PATH}")
    
    # 무결성 검증 (Graph의 입출력 짝이 맞는지 검사)
    resnet_onnx = onnx.load(RESNET_ONNX_PATH)
    onnx.checker.check_model(resnet_onnx)
    print("✅ ONNX 무결성 검증 통과!")
    
    # 파일 크기 확인
    size_kb = os.path.getsize(RESNET_ONNX_PATH) / 1024
    print(f"   파일 크기: {size_kb:.2f} KB")

except Exception as e:
    print(f"❌ ONNX Export 실패: {e}")
    print("👉 Tip: 동적 제어문(if, for)이 있다면 torch.jit.script() 시도")


🔄 [Step 3] ResNet ONNX 변환
ONNX Export 시도 중...
   - Skip Connection (덧셈 연산) 포함
   - BatchNorm (eval 모드) 포함
   - Projection Shortcut (1x1 Conv) 포함

✅ ResNet ONNX Export 성공: saved_models/mini_resnet.onnx
✅ ONNX 무결성 검증 통과!
   파일 크기: 305.42 KB


C:\Users\daboi\AppData\Local\Temp\ipykernel_26884\3922738770.py:28: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


### 📊 결과 해석
- **Export 성공**: Skip Connection, BatchNorm 모두 변환됨
- **opset_version=13**: Add, BatchNormalization 등 모든 연산 지원
- **파일 크기**: SimpleCNN보다 작음 (파라미터 수 차이)

---
## 🧪 ZONE 5: 교차 검증 (PyTorch vs ONNX)

In [11]:
print_header("🧪 [Step 4] 교차 검증 (PyTorch vs ONNX)")

# ═══════════════════════════════════════════════════════════════════
# 🎲 [테스트 입력 생성]
# ═══════════════════════════════════════════════════════════════════
# Dynamic Axes가 잘 동작하는지 확인하기 위해 배치 사이즈를 3으로 설정
test_input = torch.randn(3, 1, 28, 28)

# ═══════════════════════════════════════════════════════════════════
# 1️⃣ [PyTorch 추론]
# ═══════════════════════════════════════════════════════════════════
with torch.no_grad():
    torch_output = resnet_model(test_input).numpy()

# ═══════════════════════════════════════════════════════════════════
# 2️⃣ [ONNX Runtime 추론]
# ═══════════════════════════════════════════════════════════════════
ort_session = ort.InferenceSession(
    RESNET_ONNX_PATH,
    providers=['CPUExecutionProvider']
)

ort_output = ort_session.run(
    None,
    {'input': test_input.numpy()}
)[0]

# ═══════════════════════════════════════════════════════════════════
# 3️⃣ [결과 비교]
# ═══════════════════════════════════════════════════════════════════
print(f"입력 Shape: {test_input.shape}")
print(f"출력 Shape: {torch_output.shape}")
print("-" * 50)

print("PyTorch Output (샘플 1):")
print(f"  {torch_output[0][:5]}...")
print("-" * 50)

print("ONNX Output    (샘플 1):")
print(f"  {ort_output[0][:5]}...")
print("-" * 50)

# 오차 계산
# Skip Connection(덧셈)이 많아질수록 부동소수점 오차가 누적될 수 있습니다.
is_matched = np.allclose(torch_output, ort_output, atol=1e-5)
max_diff = np.abs(torch_output - ort_output).max()

print(f"\n[Skip Connection 포함 모델 교차 검증]")
print(f"  - 최대 오차: {max_diff:.2e}")
print(f"  - 검증 결과: {'✅ 일치' if is_matched else '⚠️ 불일치'}")


🧪 [Step 4] 교차 검증 (PyTorch vs ONNX)
입력 Shape: torch.Size([3, 1, 28, 28])
출력 Shape: (3, 10)
--------------------------------------------------
PyTorch Output (샘플 1):
  [-0.10128323 -0.11644329  0.0853909   0.04903813 -0.02318882]...
--------------------------------------------------
ONNX Output    (샘플 1):
  [-0.10128323 -0.11644329  0.0853909   0.04903812 -0.02318882]...
--------------------------------------------------

[Skip Connection 포함 모델 교차 검증]
  - 최대 오차: 1.49e-08
  - 검증 결과: ✅ 일치


### 📊 결과 해석
- **Max Diff ~1e-6 ~ 1e-7**: 부동소수점 오차 수준
- **검증 성공**: Skip Connection이 ONNX에서도 정확히 동작
- **BatchNorm**: eval 모드 설정이 중요!

---
## 🔍 ZONE 6: ONNX 그래프 분석

In [12]:
print_header("🔍 [Step 5] ONNX 그래프 분석")

# ═══════════════════════════════════════════════════════════════════
# 📦 [ONNX 모델 로드]
# ═══════════════════════════════════════════════════════════════════
onnx_model = onnx.load(RESNET_ONNX_PATH)

# ═══════════════════════════════════════════════════════════════════
# 📊 [연산자(Operator) 통계]
# ═══════════════════════════════════════════════════════════════════
from collections import Counter

# [기계적 구동] 그래프의 모든 노드를 순회하며 'op_type'(연산자 종류)을 셉니다.
op_types = [node.op_type for node in onnx_model.graph.node]
op_counts = Counter(op_types)

print("[ONNX 연산자 통계]")
print("-" * 40)
for op, count in sorted(op_counts.items(), key=lambda x: -x[1]):
    print(f"  {op:<20}: {count}개")

print(f"\n총 노드 수: {len(op_types)}개")

# ═══════════════════════════════════════════════════════════════════
# 📊 [핵심 연산자 설명]
# ═══════════════════════════════════════════════════════════════════
print("\n[핵심 연산자 설명]")
print("-" * 40)
print("  Conv             : Convolution (합성곱)")
print("  Add              : ⭐ Skip Connection! (x + F(x)) 파이토치의 += 연산이 이것으로 변환됨")
print("  Relu             : ReLU 활성화")
print("  BatchNorm        : 배치 정규화 (eval 모드여서 통계값이 상수로 박힌 상태)")
print("  GlobalAvgPool    : AdaptiveAvgPool2d((1,1))이 이것으로 최적화됨")
print("  Gemm             : General Matrix Multiply (Linear 레이어)")


🔍 [Step 5] ONNX 그래프 분석
[ONNX 연산자 통계]
----------------------------------------
  Conv                : 9개
  Relu                : 7개
  Identity            : 6개
  Add                 : 3개
  GlobalAveragePool   : 1개
  Flatten             : 1개
  Gemm                : 1개

총 노드 수: 28개

[핵심 연산자 설명]
----------------------------------------
  Conv             : Convolution (합성곱)
  Add              : ⭐ Skip Connection! (x + F(x)) 파이토치의 += 연산이 이것으로 변환됨
  Relu             : ReLU 활성화
  BatchNorm        : 배치 정규화 (eval 모드여서 통계값이 상수로 박힌 상태)
  GlobalAvgPool    : AdaptiveAvgPool2d((1,1))이 이것으로 최적화됨
  Gemm             : General Matrix Multiply (Linear 레이어)


### 📊 결과 해석
- **Add 연산**: Skip Connection으로 인해 생성됨
- **BatchNormalization**: eval 모드에서 "고정된 통계" 사용
- **Flatten → Gemm**: FC 레이어로 변환됨

**핵심**: ONNX 그래프에서 Skip Connection은 **Add 노드**로 표현!

### 🏁 최종 정리

---

#### 오늘 배운 핵심

| # | 내용                | 핵심                        |
|---|---------------------|-----------------------------|
| 1 | Skip Connection     | x + F(x), 기울기 소실 완화 |
| 2 | Projection Shortcut | 1x1 Conv로 차원 맞춤        |
| 3 | BatchNorm + ONNX    | eval() 모드 필수!           |
| 4 | 복잡한 구조 변환    | 대부분 opset 13으로 충분    |


#### Skip Connection이 ONNX에서 동작하는 원리
---

#### 다음 단계

- **Part 3**: 양자화 기법 (PTQ, QAT)  
- **Part 4**: 실제 양자화 적용


### 📊 Part 2 심화: ResNet과 ONNX 결과 해석 및 분석

이 문서는 `resnet_onnx_lab.py` 코드를 실행하여 얻은 결과,  
특히 **Skip Connection과 BatchNorm이 ONNX 변환 과정에서 어떤 영향을 미치는지** 심층 분석합니다.

---

### 1. 🏗️ Residual Block과 Projection Shortcut (Step 1 해석)

✅ **왜 Projection Shortcut이 필요한가?**

ResidualBlock 테스트 결과를 보면 다음과 같은 차이가 있습니다:

- **Identity Shortcut (stride=1):**  
  - 입력: (1, 16, 28, 28)  
  - 출력: (1, 16, 28, 28)  
  - 분석: Shape이 완벽하게 같으므로, 그냥 더하기(+)만 하면 됩니다. 추가 연산 비용 0.  

- **Projection Shortcut (stride=2):**  
  - 입력: (1, 16, 28, 28)  
  - 출력: (1, 32, 14, 14)  
  - 분석: 입력과 출력의 **크기(28 vs 14)**와 **채널(16 vs 32)**이 다릅니다. 이 상태로 더하면 에러가 납니다.  
  - 해결: **1x1 Conv**를 사용해 입력을 (1, 32, 14, 14)로 강제 변환한 뒤 더합니다.  

---

### 2. 🔄 ONNX 그래프 속의 Skip Connection (Step 5 해석)

ONNX 연산자 통계를 보면 **Add 노드**가 등장합니다.

| 연산자 (Op) | 개수 (예상) | 의미 |
|-------------|-------------|------|
| Conv        | 많음        | ResNet의 핵심 연산 |
| BatchNorm   | 많음        | Conv 뒤에 항상 따라옴 |
| Add         | 3개         | 이것이 바로 Skip Connection! |

💡 **Add 노드의 정체**  
- PyTorch 코드의 `out += identity` 한 줄이 ONNX 그래프에서는 하나의 **Add 노드**가 됩니다.  
- 입력 1: Main Path (Conv → BN → ReLU → Conv → BN)의 결과  
- 입력 2: Shortcut Path (Identity 또는 Projection)의 결과  
- 출력: 두 텐서의 **요소별 합 (Element-wise Sum)**  

이 구조 덕분에 ONNX Runtime에서도 ResNet 구조가 완벽하게 돌아가는 것입니다.

---

### 3. ⚠️ BatchNorm의 함정: 왜 eval()인가?

코드에서 `resnet_model.eval()`을 강조했습니다.  
만약 `train()` 모드로 Export하면 무슨 일이 벌어질까요?

| 모드       | 동작 방식                  | ONNX 변환 결과 |
|------------|----------------------------|----------------|
| Train 모드 | 배치마다 평균/분산 계산     | 평균/분산을 계산하는 복잡한 서브 그래프들이 생성됨 → 추론 속도 느려짐 |
| Eval 모드  | 저장된 이동평균(Running Stats) 사용 | 평균/분산이 상수로 고정됨 → 단순히 `(x - mean) / std` 수식 하나로 최적화됨 |

**결론:** 배포용(ONNX)으로 변환할 때는 **무조건 eval() 모드**여야 합니다.

---

### 4. 🚀 교차 검증 결과 (Step 4 해석)

- **최대 오차(Max Diff):** 약 1e-6 ~ 1e-7  
- **의미:** PyTorch와 ONNX Runtime의 결과가 사실상 동일합니다.  
- **중요성:** Skip Connection이나 Projection 연산이 ONNX로 변환되는 과정에서 정보 손실이나 왜곡이 발생하지 않았음을 증명합니다.  

---

### 5. ✅ 최종 요약 (Takeaway)

- **Skip Connection = Add Node:** ResNet의 핵심은 ONNX에서 Add 연산자로 표현된다.  
- **Shape이 생명:** 더하기 연산을 하려면 Projection Shortcut으로 차원을 맞춰야 한다.  
- **Eval 모드 필수:** BatchNorm을 상수로 박제하기 위해 `model.eval()`을 반드시 사용해야 한다.  